In this tutorial, we will be making requests to the Anthropic API
(this content is derived from Anthropic skilljar site)

Install necessary dependencies

In [ ]:
%pip install anthropic python-dotenv

Create a .env file in the same directory. This is to store your API key securely to avoid hardcoding and accidentally committing to a version control such as Github. After that, you should add the .env file to the .gitignore

If you're using Google Colab, you can just save the API key in the **Secrets** section.

In [ ]:
ANTHROPIC_API_KEY="your api key"

Load the environment variables and create the API client by calling the Anthropic object. You can also set the model type.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

Making the first request. When you run the following code, Claude processes your request and return a response object containing the generated text along with metadata about the request

In [ ]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
          "role": "user",
          "content": "What is prompt injection? Answer in one sentence"
        }
    ]
)

The response object contains a lot of information but generally only the generated text matters. We can access it by running the following line of code

In [ ]:
message.content[0].text

So it should give a clean, readable output like "Prompt injection is a technique where untrusted input is crafted to manipulate an AI model into ignoring or overriding its intended instructions or behavior."

However, there are exceptions especially if there is a bug within the setup or a problem from the model's side (eg: prompt got blocked by a security guardrail), running the above line might crash or give an unexpected error.

It's also important to note that Claude doesn't store any of the conversation hisoty as each request is treated as independent with no memory of previous messages.

If we want Claude to remember context from earlier messages, we need to handle the conversation state overselves.

In [1]:
def add_user_message(messages, text):
  user_message = {"role": "user", "content": text}
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {"role": "assistant", "content": text}
  messages.append(assistant_message)

def chat(messages):
  message = client.messages.create(
      model=model,
      max_token=1000,
      messages = messages
  )
  return message.content[0].text

The above are three helper functions that handle conversation management for multi-turn conversations with Claude. The following is how we will use them to maintain a conversation.

In [ ]:
messages = []
add_user_message(messages, "Define prompt injection in one sentence")
answer = chat(messages)
add_assistant_message(messages, answer)
add_user_message(messages, "tell me more about it")
final_answer = chat(messages)


The following code is to provide Claude with a system prompt.
System prompts is to let Claude know how to respond to the user input. We can use system prompts to shape Claude's tone, style and approach to match a specific use case.

In [ ]:
system prompt = """
You are an expert in cybersecurity. Provide helpful
and safe responses related using best security and
privacy practices
"""

client.messages.create(
    model=model,
    messages=messages,
    max_tokens=1000,
    system=system_prompt
)

To make it more flexible without hard-coding the system prompts, we can use parameters

In [ ]:
def chat(messages, system=None):
  params = {
      "model": model,
      "max_tokens": 1000,
      "messages": messages,
  }

  if system:
    params["system"] = system

  message = client.messages.create(**params)
  return message.content[0].text


answer = chat(messages)

system = """
You are an expert in cybersecurity. Provide helpful
and safe responses related using best security and
privacy practices
"""

answer = chat(messages, system=system)

So how does Claude generates text? When we send Claude a prompt like "How are you?", it goes through the following steps:
- Tokenization -> Breaking user input into smaller chunks
- Prediction -> Calculating probabilities for possible next words
- Sampling -> Choosing a token based on those probabilities

Temperature is a decimal value between 0 and 1 that influences these selection probabilities.
A low temp produces more deterministic output (eg: one word gets assigned 100%)
A high temp produces more non-deterministic/random output (eg: probability gets distributed more evenly across options)

Pick Low Temperature (0.0 - 0.3)
- Factual responses
- Coding assistance
- Data extraction
- Content moderation

Pick Medium Temperature (0.4 - 0.7)
- Summarization
- Educational content
- Problem solving
- Creating writing with constraints

Pick High Temperature (0.8 - 1.0)
- Brainstorm
- Creative writing
- Marketing content
- Joke generation

The following is how we define the temp in code

In [ ]:
def chat(messages, system=None, temperature=1.0):
  params = {
      "model": model,
      "max_tokens": 1000,
      "messages": messages,
      "temperature": temperature
  }

When building chat apps with Claude, there's a significant user experience issue where responses can take 10-30 secs to generate. This is not ideal.

The solution is to use response streaming. This is displaying text chunk by chunk when Claude generates it, creating a more responsive feel for users.

The following is how we do it using the stream parameter which will return an array of response objects

In [ ]:
messages = []
add_user_message(messages, "Write 1 sentence description about RAG poisoning")
stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for text in stream.text_stream:
  print(text)

After streaming is done, we need to save the complete message for storage or further processing.

In [ ]:
final_message = stream.get_final_message()

Sometimes, when we need Claude to generate structured data like JSON, Python code or formatted lists, we will often run into a common problem where Claude tries to be helpful and adds explanatory text around the generated the content.

Solution is to use Assistant Message Prefilling + Stop Sequences for example the below, where we trick Claude to think that it is already generating the json content. When Claude tries to close the code block with ```. the stop sequence immediately ends the generation.

In [ ]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

The key here is to identify what Claude naturally wants to wrap your content in whether it is Python code, bulleted lists, csv data, etc...then using that as your prefill and stop sequence.